# `pogg.semantic_composition.sement_util`

The `sement_util` module contains a number of functions that are useful for manipulating, comparing, and printing SEMENT structures.

##

In [ ]:
import pprint
import pogg.my_delphin.sementcodecs as sementcodecs
import pogg.semantic_composition.sement_util as sement_util

## `group_equalities` example

Group equalities from a list of EQ pairs into EQ sets. That is, if the EQ list contains an eq `(x1, x2)` and an eq `(x2, x3)` then create a set `(x1, x2, x3)` such that they're in an equality "group"

In [ ]:
eqs = [("x1", "x2"), ("x3", "x4"), ("x1", "x4"), ("x5", "x6")]
sement_util.group_equalities(eqs)

## `get_most_specified_variable` example

Get the most "specific" variable to serve as the representative for the EQ set. That is, a variable of type `x` is more specific than one of type `i`, according to the ERG hierarchy

### ERG Variable Type Hierarchy
| Type | Supertype(s) | Description |
|------|--------------|-------------|
| `u`  |              | unspecific |
| `i`  | `u`          | underspecified between `e` and `x` |
| `p`  | `u`          | underspecified between `h` and `x` |
| `e`  | `i`          | eventualities (e.g. intrinsic variable of a verb) |
| `x`  | `i`, `p`     | instance (e.g. intrinsic variable of a noun) |
| `h`  | `p`          | handle used for scopal composition |



In [ ]:
vars = ["u1", "i2", "x3"]
sement_util.get_most_specified_variable(vars)

## `overwrite_eqs` example

Example of `overwrite_eqs` being called on a SEMENT for *tasty cookie*. In the initial SEMENT, there is an EQ between the `ARG0` of *cookie* (`x1`) and the `ARG1` of *tasty*, because the `ARG1` of *cookie* (i.e. the thing that is tasty) is plugged by the intrinsic variable of *cookie*.

The `overwrite_eqs` function chooses one of these variables as the representative for the EQ (here, `x1`) and overwrites all instances of `x4` to also be `x1`. This enables conversion to an MRS object in order to send the structure to the ERG for text generation.

In [ ]:
# SEMENT for "tasty cookie" before EQs are overwritten
sement_string = """[
    TOP: h0
    INDEX: x1
    RELS: <
        [ _tasty_a_1 LBL: h2 ARG0: e3 ARG1: x4 ]
        [ _cookie_n_1 LBL: h0 ARG0: x1 ] >
    EQS: < x1 eq x4 >
]"""

# convert string into SEMENT object
original_sement = sementcodecs.decode(sement_string)

new_sement = sement_util.overwrite_eqs(original_sement)

# encode the new_sement object into a string and print it
print(sementcodecs.encode(new_sement, indent=True))

## `check_if_quantified` example

This function is used in cases where it's necessary to ensure that a SEMENT is quantified before proceeding with composition. For example, the argument of a verb must be plugged with a quantified noun (plus possible adjuncts). That is, it cannot be plugged with a SEMENT whose `RELS` list only contains one EP for *cookie*, but a SEMENT whose `RELS` list contains an EP for *cookie* and some quantifier.

In [ ]:
unquant_cookie = """[ TOP: h0
    INDEX: x1
    RELS: < [ _cookie_n_1 LBL: h0 ARG0: x1 ] >
]"""
unquant_cookie_sement_obj = sementcodecs.decode(unquant_cookie)

sement_util.check_if_quantified(unquant_cookie_sement_obj)

In [ ]:
quant_cookie = """[ TOP: h6
    INDEX: x1
    RELS: <
        [ _udef_q LBL: h0 ARG0: x1 RSTR: h2 BODY: h3 ]
        [ _cookie_n_1 LBL: h4 ARG0: x5 ] >
    EQS: < x1 eq x5 >
    SLOTS: < BODY: h3 >
    HCONS: < h2 qeq h4 > ]"""

quant_cookie_sement_obj = sementcodecs.decode(quant_cookie)

sement_util.check_if_quantified(quant_cookie_sement_obj)

## `is_sement_isomorphic` example

Here are two examples of `is_sement_isomorphic`. In the first one, the SEMENTs are isomorphic, but just have different variable names and different orders on the `RELS` lists, so the check returns `True`.

In [ ]:
sement1 = """[
    TOP: h0101
    INDEX: e1101 [e NUM: sg]
    RELS: <
        [ _a_q LBL: h10 ARG0: x11 RSTR: h12 BODY: h13]
        [ _cookie_n_1 LBL: h6 ARG0: x7 ]
        [ _give_v_1 LBL: h0 ARG0: e1 ARG1: i2 ARG2: u3 ARG3: i4 ]
    >
    EQS: < x11 eq x7 eq i2 h0101 eq h0 e1 eq e1101 >
    SLOTS: < ARG2: u3 ARG3: i4 >
    HCONS: < h12 qeq h6 >

]"""

sement2 = """[
    TOP: h011
    INDEX: e111 [e NUM: sg]
    RELS: <
        [ _give_v_1 LBL: h011 ARG0: e111 ARG1: i211 ARG2: u311 ARG3: i411 ]
        [ _a_q LBL: h1011 ARG0: x1111 RSTR: h1211 BODY: h1311]
        [ _cookie_n_1 LBL: h611 ARG0: x711 ]
    >
    EQS: < x1111 eq x711 x711 eq i211 >
    SLOTS: < ARG2: u311 ARG3: i411 >
    HCONS: < h1211 qeq h611 >

]"""

sement1_obj = sementcodecs.decode(sement1)
sement2_obj = sementcodecs.decode(sement2)

sement_util.is_sement_isomorphic(sement1_obj, sement2_obj)

In the second example, the SEMENTs are almost isomorphic, but the `TOP` is identified with the `LBL` of the `_cookie_n_1` EP, rather than the `LBL` of the `_give_v_1` EP, causing a discrepancy.

In [ ]:
sement3 = """[
    TOP: h611
    INDEX: e111 [e NUM: sg]
    RELS: <
        [ _give_v_1 LBL: h011 ARG0: e111 ARG1: i211 ARG2: u311 ARG3: i411 ]
        [ _a_q LBL: h1011 ARG0: x1111 RSTR: h1211 BODY: h1311]
        [ _cookie_n_1 LBL: h611 ARG0: x711 ]
    >
    EQS: < x1111 eq x711 x711 eq i2 >
    SLOTS: < ARG2: u311 ARG3: i411 >
    HCONS: < h1211 qeq h611 >

]"""

sement3_obj = sementcodecs.decode(sement3)

# compare again to the first SEMENT
sement_util.is_sement_isomorphic(sement1_obj, sement3_obj)

Detecting these differences at a glance is difficult, and the function `build_isomorphism_report` aims to alleviate this, but before going over an example of that a number of helper functions are used within `build_isomorphism_report` so let's go over those first.

## `create_variable_roles_dict` example

Given a SEMENT object, create a dictionary where each key is a variable in the SEMENT and the value is the set of semantic roles that variable fills. Having this information will help when comparing two SEMENTs.

Below is an example of calling it on a SEMENT for *a tasty cookie*



In [ ]:
sement_string = """[ TOP: h9
                  INDEX: x4
                  RELS: < [ _tasty_a_1 LBL: h0 ARG0: e1 ARG1: x4 ]
                          [ _cookie_n_1 LBL: h0 ARG0: x4 ]
                          [ _a_q LBL: h5 ARG0: x4 RSTR: h7 BODY: h8 ] >
                  SLOTS: < BODY: h8 >
                  HCONS: < h7 qeq h0 > ]"""

sement_object = sementcodecs.decode(sement_string)

roles_dict = sement_util.create_variable_roles_dict(sement_object)

pprint.pp(roles_dict)

## `create_hcons_list` example

Create a list of HCons entries. Each entry includes the handles that are in the HCons relationship as well as the semantic roles those handles occupy to enable easier comparison of existing handle constraints across SEMENTs.



In [ ]:
sement_string = """[ TOP: h9
  INDEX: x4
  RELS: < [ _tasty_a_1 LBL: h0 ARG0: e1 ARG1: x4 ]
          [ _cookie_n_1 LBL: h0 ARG0: x4 ]
          [ _a_q LBL: h5 ARG0: x4 RSTR: h7 BODY: h8 ] >
  SLOTS: < BODY: h8 >
  HCONS: < h7 qeq h0 > ]"""

sement_object = sementcodecs.decode(sement_string)

hcons_list = sement_util.create_hcons_list(sement_object)
pprint.pp(hcons_list)

## `find_slot_overlaps` example

Produces three lists: `overlap_slots`, `gold_only_slots`, and `actual_only_slots`. The goal is to compare slots lists across two SEMENTs to detect differences when isomorphism checks fail.

Each list contains dictionaries that detail the slots that remain.

```
    {
        "slot": {"_cozy_a_1.ARG1"}
        "gold_var": 'x1',
        "actual_var": 'x2'
    }
```

If two SEMENts are isomorphic, the `gold_only_slots` and `actual_only_slots` lists will be empty, but when the SEMENTs are not isomorphic, the sets of remaining slots may not match so these lists will help pinpoint where the differences lie.

### Example

Below is an example of calling :py:func:`find_slot_overlaps` on two SEMENTs for *believe the cat sleeps*. In the broken one, there is a handle constraint between the `ARG3` of the `_believe_v_1` EP and the `LBL` of the `_sleep_v_1` EP, which is the incorrect slot for this constraint, causing the slots list to be incorrect. [^bignote]

[^bignote]: Technically, they are both broken because we want the two argument version of `_believe_v_1`, not the three argument version, but using the three argument version is better for this example.

In [ ]:
gold_sement_str = """[ TOP: h0
  INDEX: e1
  RELS: < [ _believe_v_1 LBL: h0 ARG0: e1 ARG1: i2 ARG2: u3 ARG3: h4 ]
          [ _the_q LBL: h5 ARG0: x10 RSTR: h7 BODY: h8 ]
          [ _cat_n_1 LBL: h9 ARG0: x10 ]
          [ _sleep_v_1 LBL: h11 ARG0: e12 ARG1: x10 ] >
  SLOTS: < ARG1: i2 ARG3: h4 >
  HCONS: < h7 qeq h9 u3 qeq h11 > ]"""

broken_sement_str = """[ TOP: h00
  INDEX: e01
  RELS: < [ _believe_v_1 LBL: h00 ARG0: e01 ARG1: i02 ARG2: u03 ARG3: h04 ]
          [ _the_q LBL: h05 ARG0: x010 RSTR: h07 BODY: h08 ]
          [ _cat_n_1 LBL: h09 ARG0: x010 ]
          [ _sleep_v_1 LBL: h011 ARG0: e012 ARG1: x010 ] >
  SLOTS: < ARG1: i02 ARG2: u03 >
  HCONS: < h07 qeq h09 h04 qeq h011 > ]"""

gold_sement_obj = sementcodecs.decode(gold_sement_str)
broken_sement_obj = sementcodecs.decode(broken_sement_str)

# the function returns three lists, store each of them
overlapping_slots, gold_slots, broken_slots = sement_util.find_slot_overlaps(gold_sement_obj, broken_sement_obj)


# print resulting slot lists
print("OVERLAPPING: {}".format(overlapping_slots))
print("GOLD: {}".format(gold_slots))
print("BROKEN: {}".format(broken_slots))

## `find_var_eq_overlaps` example

Produces three lists: `overlap_eqs`, `gold_only_eqs`, and `actual_only_eqs`. The goal is to compare semantic role identities across two SEMENTs to detect differences when isomorphism checks fail.

Each list contains dictionaries that detail sets of semantic roles that are filled by the same variable.


Here is an example of a "role equivalency set":

```
    {
        "eq_set": {"_a_q.ARG0", "_cat_n_1.ARG0", "_cozy_a_1.ARG1"}
        "gold_var": 'x1',
        "actual_var": 'x2'
    }
```

What this shows is that in both SEMENTs the `ARG0` of the `_a_q` EP, the `ARG0` of the `_cat_n_1` EP, and the `ARG1` of the `_cozy_a_1` EP are identified with each other, but that in the gold SEMENT the variable filling all these slots is called `x1` but in the actual (where "actual" roughly means the one being composed with the goal to match the gold) the variable is called `x2`. So, despite the different variable names, the semantic role equivalency matches.

If two SEMENTs are isomorphic, the `gold_only_eqs` and `actual_only_eqs` lists will be empty, but when the SEMENTs are not isomorphic, the sets of semantic role identities may not match so these lists will help pinpoint where the differences lie.

### Example


Below is a worked example of calling `find_var_eq_overlaps` on two SEMENTs for *tasty cookie*. In the broken one, there should be an equivalence between the `LBL` of each EP, but it's missing.

First, let's gather the lists of role equivalence sets

In [16]:
gold_tasty_cookie = """[ TOP: h0
                  INDEX: x4
                  RELS: < [ _tasty_a_1 LBL: h0 ARG0: e1 ARG1: x4 ]
                          [ _cookie_n_1 LBL: h0 ARG0: x4 ] > ]"""

broken_tasty_cookie = """[ TOP: h03
  INDEX: x04
  RELS: < [ _tasty_a_1 LBL: h00 ARG0: e01 ARG1: x04 ]
          [ _cookie_n_1 LBL: h03 ARG0: x04 ] > ]"""

gold_sement_obj = sementcodecs.decode(gold_tasty_cookie)
broken_sement_obj = sementcodecs.decode(broken_tasty_cookie)

# the function returns three lists, store each of them
overlapping_eqs, gold_eqs, broken_eqs = sement_util.find_var_eq_overlaps(gold_sement_obj, broken_sement_obj)

Now, let's look at the overlapping sets. Notice that the first set has multiple semantic roles in it (the `INDEX`, the `ARG0` for the `_cookie_n_1` EP, and the `ARG1` for the `_tasty_a_1` EP). But the second set just contains one role, the `ARG0` for the `_tasty_a_1` EP. Even though it feels odd to call that a set of "identities" it does indicate to us that in both SEMENTs this role is not identified with any other role.

In [17]:
# shows the eqs present in both SEMENTS
pprint.pp(overlapping_eqs)

[{'eq_set': ['INDEX', '_cookie_n_1.ARG0', '_tasty_a_1.ARG1'],
  'gold_var': 'x4',
  'actual_var': 'x04'},
 {'eq_set': ['_tasty_a_1.ARG0'], 'gold_var': 'e1', 'actual_var': 'e01'}]


Next, let's look at the role identities only present in the gold SEMENT. Here, we see the identity between the `LBL` of each EP as mentioned above, as well as the fact that the `TOP` is a member of this set.

In [18]:
# shows the eqs present in only in the gold SEMENT
pprint.pp(gold_eqs)

[{'eq_set': ['TOP', '_cookie_n_1.LBL', '_tasty_a_1.LBL'], 'gold_var': 'h0'}]


Last, let's look at the role identities unique to the broken SEMENT. Here, we have two sets. The first one shows that the `TOP`` is identified with the `LBL`` of the `_cookie_n_1` EP. The second one shows that the `LBL` of the `_tasty_a_1` EP is not identified with anything, which is in obvious contrast to what we saw in the gold one.

In [19]:
# shows the eqs present in only in the "actual" SEMENT
pprint.pp(broken_eqs)

[{'eq_set': ['TOP', '_cookie_n_1.LBL'], 'actual_var': 'h03'},
 {'eq_set': ['_tasty_a_1.LBL'], 'actual_var': 'h00'}]


# `find_hcons_overlaps` example

Produces three lists: `overlap_hcons`, `gold_only_hcons`, and `actual_only_hcons`. The goal is to compare handle constraints across two SEMENTs to detect differences when isomorphism checks fail.

Each list contains dictionaries that detail which handle constraints are present in which SEMENT.

Here is an example of a "role equivalency set":
```

    {
        "hi_role_set": {"_a_q.RSTR"},
        "lo_role_set": {"_cookie_n_1.LBL", "_tasty_a_1.LBL"},
        "hi_gold_var": "h0",
        "lo_gold_var": "h1",
        "hi_actual_var": "h00",
        "lo_actual_var": "h01",
    }
```

What this shows is that in both SEMENTs the `RSTR` of the `_a_q` EP is the high handle in a QEQ relationship to the `LBL` of both `_cookie_n_1` and `_tasty_a_1`, which serve as the low handles in conjunction. But the values of the handles in each SEMENT are different, even though the shape of the handle constraint is the same in both.

If two SEMENts are isomorphic, the `gold_only_hcons` and `actual_only_hcons` lists will be empty, but when the SEMENTs are not isomorphic, the sets of handle constraints may not match so these lists will help pinpoint where the differences lie.

### Example

Below is a worked example of calling `find_var_eq_overlaps` on two SEMENTs for *believe the cat sleeps*. In the broken one, there should be a QEQ between the `ARG2` of the `_believe_v_1` EP and the `LBL` of the `_sleep_v_1` EP, but it's actually set between the wrong arguments. [^bignote2]

[^bignote2]: Technically, they are both broken because we want the two argument version of `_believe_v_1`, not the three argument version, but using the three argument version is better for this example.

First, let's gather the lists of handle constraints.



In [20]:
believe_the_cat_sleeps = """[ TOP: h0
  INDEX: e1
  RELS: < [ _believe_v_1 LBL: h0 ARG0: e1 ARG1: i2 ARG2: u3 ARG3: h4 ]
          [ the_q LBL: h5 ARG0: x10 RSTR: h7 BODY: h8 ]
          [ _cat_n_1 LBL: h9 ARG0: x10 ]
          [ _sleep_v_1 LBL: h11 ARG0: e12 ARG1: x10 ] >
  SLOTS: < ARG1: i2 ARG3: h4 >
  HCONS: < h7 qeq h9 u3 qeq h11 > ]"""

broken_believe_the_cat_sleeps = """[ TOP: h00
  INDEX: e01
  RELS: < [ _believe_v_1 LBL: h00 ARG0: e01 ARG1: i02 ARG2: u03 ARG3: h04 ]
          [ the_q LBL: h05 ARG0: x010 RSTR: h07 BODY: h08 ]
          [ _cat_n_1 LBL: h09 ARG0: x010 ]
          [ _sleep_v_1 LBL: h011 ARG0: e012 ARG1: x010 ] >
  SLOTS: < ARG1: i02 >
  HCONS: < h07 qeq h09 h04 qeq h011 > ]"""

gold_sement_obj = sementcodecs.decode(believe_the_cat_sleeps)
broken_sement_obj = sementcodecs.decode(broken_believe_the_cat_sleeps)

overlapping_hcons, gold_hcons, broken_hcons = (sement_util.find_hcons_overlaps(gold_sement_obj, broken_sement_obj))

Now, let's look at the overlapping handle constraints. Here we see that in both SEMENTs we have the QEQ relationship between the `RSTR` of the quantifier EP (`_the_q`) and the `LBL` of the `_cat_n_1` EP.

In [21]:
# shows the hcons present in both SEMENTS
pprint.pp(overlapping_hcons)

[{'hi_role_set': ['the_q.RSTR'],
  'lo_role_set': ['_cat_n_1.LBL'],
  'gold_hi_var': 'h7',
  'gold_lo_var': 'h9',
  'actual_hi_var': 'h07',
  'actual_lo_var': 'h09'}]


Next, let's look at the handle constraints only present in the gold SEMENT. Here, we see the desired QEQ relationship between the `ARG2` of the `_believe_v_1` EP and the `LBL` of the `_sleep_v_1` EP.

In [22]:
# shows the hcons present in only in the gold SEMENT
pprint.pp(gold_hcons)

[{'hi_role_set': ['_believe_v_1.ARG2'],
  'lo_role_set': ['_sleep_v_1.LBL'],
  'gold_hi_var': 'u3',
  'gold_lo_var': 'h11'}]


Last, let's look at the handle constraints unique to the broken SEMENT. Here, we have two sets. Here we see the error in the broken version: the high handle in the QEQ is the `ARG3` of the `_believe_v_1` EP rather than the `ARG2`.

In [23]:
# shows the hcons present in only in the "actual" SEMENT
pprint.pp(broken_hcons)

[{'hi_role_set': ['_believe_v_1.ARG3'],
  'lo_role_set': ['_sleep_v_1.LBL'],
  'actual_hi_var': 'h04',
  'actual_lo_var': 'h011'}]
